# Hacker News Data Analysis

Analysis of scraped Hacker News stories from our dbt mart tables.

**Prerequisites:** Run the ingestion pipeline and dbt models first:
```bash
make ingest
make dbt-build
```

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

load_dotenv()

PROJECT = os.getenv("GCP_PROJECT_ID", "your-gcp-project-id")
client = bigquery.Client(project=PROJECT)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load mart data from BigQuery

In [ ]:
stories_df = client.query("""
    SELECT * FROM `data_sandbox_dev_marts.fct_hackernews_stories`
""").to_dataframe()

authors_df = client.query("""
    SELECT * FROM `data_sandbox_dev_marts.fct_hackernews_authors`
""").to_dataframe()

domains_df = client.query("""
    SELECT * FROM `data_sandbox_dev_marts.fct_hackernews_domains`
""").to_dataframe()

print(f"Stories: {len(stories_df):,}")
print(f"Authors: {len(authors_df):,}")
print(f"Domains: {len(domains_df):,}")
stories_df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Score distribution
stories_df["score"].hist(bins=30, ax=axes[0], color="#ff6600", edgecolor="white")
axes[0].set_title("Score Distribution")
axes[0].set_xlabel("Score")

# Comment distribution
stories_df["comment_count"].hist(bins=30, ax=axes[1], color="#ff6600", edgecolor="white")
axes[1].set_title("Comment Count Distribution")
axes[1].set_xlabel("Comments")

# Engagement score distribution
stories_df["engagement_score"].hist(bins=30, ax=axes[2], color="#ff6600", edgecolor="white")
axes[2].set_title("Engagement Score Distribution")
axes[2].set_xlabel("Engagement")

plt.tight_layout()
plt.show()

## 3. Top Domains & Story Types

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 domains by story count
top_domains = domains_df.nlargest(15, "total_stories")
axes[0].barh(top_domains["domain"], top_domains["total_stories"], color="#ff6600")
axes[0].set_title("Top 15 Domains by Story Count")
axes[0].invert_yaxis()

# Story type breakdown
type_counts = stories_df["story_type"].value_counts()
type_counts.plot.pie(ax=axes[1], autopct="%1.1f%%", colors=plt.cm.Oranges([0.3, 0.4, 0.5, 0.6, 0.7, 0.8]))
axes[1].set_title("Story Type Breakdown")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## 4. Simple Predictive Model — Score Prediction

Train a basic model to predict story score based on available features.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

model_df = stories_df[["score", "comment_count", "story_type", "scraped_hour", "scraped_dow"]].dropna()

le = LabelEncoder()
model_df = model_df.copy()
model_df["story_type_encoded"] = le.fit_transform(model_df["story_type"])

features = ["comment_count", "story_type_encoded", "scraped_hour", "scraped_dow"]
X = model_df[features]
y = model_df["score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f"R² Score: {r2_score(y_test, y_pred):.3f}")
print(f"MAE:      {mean_absolute_error(y_test, y_pred):.1f}")

In [ ]:
# Feature importance
importance = pd.Series(rf.feature_importances_, index=features).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
importance.plot.barh(ax=ax, color="#ff6600")
ax.set_title("Feature Importance for Score Prediction")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted scatter
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred, alpha=0.4, color="#ff6600", edgecolors="white", s=30)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "k--", lw=2)
ax.set_xlabel("Actual Score")
ax.set_ylabel("Predicted Score")
ax.set_title("Actual vs Predicted Score")
plt.tight_layout()
plt.show()